# Sample Training – OCR-Robust Multilingual Embeddings

Two-stage fine-tuning of `Alibaba-NLP/gte-multilingual-base`:

| Stage | Task | Data |
|-------|------|------|
| **A** | Cross-lingual alignment (no noise) | Luxembourgish parallel pairs (`LREC/luxembourgish_dataset/`) |
| **B** | OCR-noise robustness | ACL + LREC noisy finetuning CSVs |

**Hyperparameters:**
- `batch_size = 8`
- `batches_per_epoch = 3000`
- `epochs = 1`
- `lr = 2e-5`
- `seed = 42`
- `loss = MultipleNegativesRankingLoss`
- `warmup = 10% of steps`

## 0. Setup

In [ ]:
import os, json, random, math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers import util as st_util

MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"
BATCH_SIZE = 8
BATCHES_PER_EPOCH = 3000
EXAMPLES_PER_EPOCH = BATCH_SIZE * BATCHES_PER_EPOCH  # 24 000
EPOCHS = 1
LR = 2e-5
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Evaluate BEFORE training

In [ ]:
def clsd_accuracy(model, csv_path, src_col, tgt_col):
    """Compute CLSD retrieval accuracy (source → target)."""
    df = pd.read_csv(csv_path).dropna(subset=[src_col, tgt_col])
    src = df[src_col].astype(str).tolist()
    tgt = df[tgt_col].astype(str).tolist()

    src_emb = model.encode(src, convert_to_tensor=True, show_progress_bar=False)
    tgt_emb = model.encode(tgt, convert_to_tensor=True, show_progress_bar=False)

    sims = st_util.cos_sim(src_emb, tgt_emb)
    preds = sims.argmax(dim=1).cpu()
    gold = torch.arange(len(src))
    return (preds == gold).float().mean().item()


EVAL_CLEAN = "ACL/noisy_evaluation_datasets/CLSD_wmt2019_adversarial_dataset.csv"
EVAL_NOISY = "ACL/noisy_evaluation_datasets/CLSD_WMT19_SNP_noise.csv"

base_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True).to(device)

acc_clean_before = clsd_accuracy(base_model, EVAL_CLEAN, "fra", "deu")
acc_noisy_before = clsd_accuracy(base_model, EVAL_NOISY, "fra_04", "deu_04")

del base_model; torch.cuda.empty_cache()

print(f"{'Condition':<12} {'Accuracy':>8}")
print(f"{'-'*12} {'-'*8}")
print(f"{'Clean':<12} {acc_clean_before:>8.4f}")
print(f"{'Noisy':<12} {acc_noisy_before:>8.4f}")

## 2. Load training data

In [ ]:
# ---------- Task A: Luxembourgish parallel pairs ----------
def load_lux_pairs(path, tgt_lang):
    """Load JSONL → list of (lb, tgt) tuples."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            for t in obj["translation"]:
                pairs.append((t["lb"], t[tgt_lang]))
    return pairs

lux_de = load_lux_pairs("LREC/luxembourgish_dataset/lb_de_training_set.jsonl", "de")
lux_en = load_lux_pairs("LREC/luxembourgish_dataset/lb_en_training_set.jsonl", "en")
lux_fr = load_lux_pairs("LREC/luxembourgish_dataset/lb_fr_training_set.jsonl", "fr")

task_a_pairs = lux_de + lux_en + lux_fr
print(f"Task A pairs: {len(task_a_pairs):,}")

In [ ]:
# ---------- Task B: OCR-noised clean/noisy pairs ----------
def csv_pairs(path, clean_col, noisy_col):
    df = pd.read_csv(path).dropna(subset=[clean_col, noisy_col])
    return list(zip(df[clean_col].astype(str), df[noisy_col].astype(str)))

task_b_pairs = []

# ACL finetuning data
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/TED_data_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/TED_data_random_noise.csv", "fra", "fra_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/X-News_data_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/X-News_data_random_noise.csv", "fra", "fra_04")

# LREC finetuning data (historical articles)
task_b_pairs += csv_pairs("LREC/noisy_finetuning_data/de_docs_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("LREC/noisy_finetuning_data/fr_docs_random_noise.csv", "fra", "fra_04")

print(f"Task B pairs: {len(task_b_pairs):,}")

In [ ]:
def pairs_to_examples(pairs, n):
    """Sample *n* InputExamples (with replacement if needed)."""
    rng = random.Random(SEED)
    sampled = [pairs[rng.randrange(len(pairs))] for _ in range(n)]
    return [InputExample(texts=[a, b]) for a, b in sampled]

examples_a = pairs_to_examples(task_a_pairs, EXAMPLES_PER_EPOCH)
examples_b = pairs_to_examples(task_b_pairs, EXAMPLES_PER_EPOCH)
print(f"Stage A examples: {len(examples_a):,}  |  Stage B examples: {len(examples_b):,}")

## 3. Stage A – Cross-lingual alignment

In [ ]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True).to(device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

loader_a = DataLoader(examples_a, batch_size=BATCH_SIZE, shuffle=False)
warmup_a = int(0.1 * len(loader_a))

print(f"Stage A: {len(loader_a)} batches, warmup={warmup_a}")

model.fit(
    train_objectives=[(loader_a, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_a,
    optimizer_params={"lr": LR},
    use_amp=True,
    output_path="trained_models/stage_a",
    show_progress_bar=True,
)

## 4. Stage B – OCR-noise robustness

In [ ]:
# Continue from Stage A checkpoint
model = SentenceTransformer("trained_models/stage_a", trust_remote_code=True).to(device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

loader_b = DataLoader(examples_b, batch_size=BATCH_SIZE, shuffle=False)
warmup_b = int(0.1 * len(loader_b))

print(f"Stage B: {len(loader_b)} batches, warmup={warmup_b}")

model.fit(
    train_objectives=[(loader_b, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_b,
    optimizer_params={"lr": LR},
    use_amp=True,
    output_path="trained_models/stage_b",
    show_progress_bar=True,
)

## 5. Evaluate AFTER training

In [ ]:
ft_model = SentenceTransformer("trained_models/stage_b", trust_remote_code=True).to(device)

acc_clean_after = clsd_accuracy(ft_model, EVAL_CLEAN, "fra", "deu")
acc_noisy_after = clsd_accuracy(ft_model, EVAL_NOISY, "fra_04", "deu_04")

print(f"{'Condition':<12} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"{'-'*12} {'-'*8} {'-'*8} {'-'*8}")
print(f"{'Clean':<12} {acc_clean_before:>8.4f} {acc_clean_after:>8.4f} {acc_clean_after - acc_clean_before:>+8.4f}")
print(f"{'Noisy':<12} {acc_noisy_before:>8.4f} {acc_noisy_after:>8.4f} {acc_noisy_after - acc_noisy_before:>+8.4f}")

## 6. Save final model

In [ ]:
final_dir = "trained_models/gte-multilingual-base-ocr-robust"
ft_model.save(final_dir)
print(f"Saved final model to {final_dir}")

# Sample Training – OCR-Robust Multilingual Embeddings

Two-stage fine-tuning of `Alibaba-NLP/gte-multilingual-base`:

| Stage | Task | Data |
|-------|------|------|
| **A** | Cross-lingual alignment (no noise) | Luxembourgish parallel pairs (`LREC/luxembourgish_dataset/`) |
| **B** | OCR-noise robustness | ACL + LREC noisy finetuning CSVs |

**Hyperparameters** (from `src/finetuning/finetuning_no_sampler_args.py`):
- `batch_size = 8`
- `batches_per_epoch = 3000`
- `epochs = 1`
- `lr = 2e-5`
- `seed = 42`
- `loss = MultipleNegativesRankingLoss`
- `warmup = 10 % of steps`

In [ ]:
import os, json, random, math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer, InputExample, losses

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load data

In [ ]:
# ---------- Task A: Luxembourgish parallel pairs ----------
def load_lux_pairs(path, tgt_lang):
    """Load JSONL → list of (lb, tgt) tuples."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            for t in obj["translation"]:
                pairs.append((t["lb"], t[tgt_lang]))
    return pairs

lux_de = load_lux_pairs("LREC/luxembourgish_dataset/lb_de_training_set.jsonl", "de")
lux_en = load_lux_pairs("LREC/luxembourgish_dataset/lb_en_training_set.jsonl", "en")
lux_fr = load_lux_pairs("LREC/luxembourgish_dataset/lb_fr_training_set.jsonl", "fr")

task_a_pairs = lux_de + lux_en + lux_fr
print(f"Task A pairs: {len(task_a_pairs):,}")

In [ ]:
# ---------- Task B: OCR-noised clean/noisy pairs ----------
def csv_pairs(path, clean_col, noisy_col):
    df = pd.read_csv(path).dropna(subset=[clean_col, noisy_col])
    return list(zip(df[clean_col].astype(str), df[noisy_col].astype(str)))

task_b_pairs = []

# ACL finetuning data
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/TED_data_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/TED_data_random_noise.csv", "fra", "fra_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/X-News_data_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("ACL/noisy_finetuning_data/X-News_data_random_noise.csv", "fra", "fra_04")

# LREC finetuning data (historical articles)
task_b_pairs += csv_pairs("LREC/noisy_finetuning_data/de_docs_random_noise.csv", "deu", "deu_04")
task_b_pairs += csv_pairs("LREC/noisy_finetuning_data/fr_docs_random_noise.csv", "fra", "fra_04")

print(f"Task B pairs: {len(task_b_pairs):,}")

## 2. Build `InputExample` lists

In [ ]:
BATCH_SIZE = 8
BATCHES_PER_EPOCH = 3000
EXAMPLES_PER_EPOCH = BATCH_SIZE * BATCHES_PER_EPOCH  # 24 000

def pairs_to_examples(pairs, n):
    """Sample *n* InputExamples (with replacement if needed)."""
    rng = random.Random(SEED)
    sampled = [pairs[rng.randrange(len(pairs))] for _ in range(n)]
    return [InputExample(texts=[a, b]) for a, b in sampled]

examples_a = pairs_to_examples(task_a_pairs, EXAMPLES_PER_EPOCH)
examples_b = pairs_to_examples(task_b_pairs, EXAMPLES_PER_EPOCH)
print(f"Stage A examples: {len(examples_a):,}  |  Stage B examples: {len(examples_b):,}")

## 3. Stage A – Cross-lingual alignment

In [ ]:
MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"
LR = 2e-5
EPOCHS = 1

model = SentenceTransformer(MODEL_NAME, trust_remote_code=True).to(device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

loader_a = DataLoader(examples_a, batch_size=BATCH_SIZE, shuffle=False)
warmup_a = int(0.1 * len(loader_a))

print(f"Stage A: {len(loader_a)} batches, warmup={warmup_a}")

model.fit(
    train_objectives=[(loader_a, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_a,
    optimizer_params={"lr": LR},
    use_amp=True,
    output_path="trained_models/stage_a",
    show_progress_bar=True,
)

## 4. Stage B – OCR-noise robustness

In [ ]:
# Continue from Stage A checkpoint
model = SentenceTransformer("trained_models/stage_a", trust_remote_code=True).to(device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

loader_b = DataLoader(examples_b, batch_size=BATCH_SIZE, shuffle=False)
warmup_b = int(0.1 * len(loader_b))

print(f"Stage B: {len(loader_b)} batches, warmup={warmup_b}")

model.fit(
    train_objectives=[(loader_b, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_b,
    optimizer_params={"lr": LR},
    use_amp=True,
    output_path="trained_models/stage_b",
    show_progress_bar=True,
)

## 5. Before / After – CLSD evaluation

In [ ]:
from sentence_transformers import util as st_util

def clsd_accuracy(model, csv_path, src_col, tgt_col, prefix=""):
    """Compute CLSD retrieval accuracy (source → target)."""
    df = pd.read_csv(csv_path).dropna(subset=[src_col, tgt_col])
    src = df[src_col].astype(str).tolist()
    tgt = df[tgt_col].astype(str).tolist()

    src_emb = model.encode([prefix + s for s in src], convert_to_tensor=True, show_progress_bar=False)
    tgt_emb = model.encode([prefix + t for t in tgt], convert_to_tensor=True, show_progress_bar=False)

    sims = st_util.cos_sim(src_emb, tgt_emb)
    preds = sims.argmax(dim=1).cpu()
    gold = torch.arange(len(src))
    return (preds == gold).float().mean().item()

eval_csv = "ACL/noisy_evaluation_datasets/CLSD_wmt2019_adversarial_dataset.csv"

# Baseline
base_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True).to(device)
acc_before = clsd_accuracy(base_model, eval_csv, "fra", "deu")
print(f"Before fine-tuning – CLSD Accuracy: {acc_before:.4f}")
del base_model; torch.cuda.empty_cache()

# After Stage B
ft_model = SentenceTransformer("trained_models/stage_b", trust_remote_code=True).to(device)
acc_after = clsd_accuracy(ft_model, eval_csv, "fra", "deu")
print(f"After fine-tuning  – CLSD Accuracy: {acc_after:.4f}")

# Noisy evaluation
noisy_csv = "ACL/noisy_evaluation_datasets/CLSD_WMT19_SNP_noise.csv"
acc_noisy_before = clsd_accuracy(base_model if 'base_model' in dir() else SentenceTransformer(MODEL_NAME, trust_remote_code=True).to(device), noisy_csv, "fra_04", "deu_04")
acc_noisy_after  = clsd_accuracy(ft_model, noisy_csv, "fra_04", "deu_04")
print(f"Noisy before: {acc_noisy_before:.4f}  |  Noisy after: {acc_noisy_after:.4f}")

## 6. Save final model

In [ ]:
final_dir = "trained_models/gte-multilingual-base-ocr-robust"
ft_model.save(final_dir)
print(f"Saved final model to {final_dir}")